# The Bias–Variance Tradeoff

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/03-bias-variance

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Simulating the decomposition

Bias and variance are defined over *many* training sets. Simulation lets us actually draw them.

In [ ]:
def true_f(x): return np.sin(2 * np.pi * x)

def knn_predict(Xtr, ytr, xq, k):
    idx = np.argsort(np.abs(Xtr[None, :] - xq[:, None]), axis=1)[:, :k]
    return ytr[idx].mean(axis=1)

x_grid = np.linspace(0, 1, 200)
def draw_dataset(n=40, noise=0.3):
    x = np.sort(np.random.rand(n)); return x, true_f(x) + noise * np.random.randn(n)

## Many datasets, two values of k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in [(axes[0], 1), (axes[1], 25)]:
    preds = []
    for _ in range(30):
        x, y = draw_dataset()
        p = knn_predict(x, y, x_grid, k); preds.append(p)
        ax.plot(x_grid, p, color='#6366f1', alpha=0.15, lw=1)
    ax.plot(x_grid, true_f(x_grid), color='#14b8a6', lw=2, label='truth')
    ax.plot(x_grid, np.mean(preds, axis=0), color='#f43f5e', lw=2, ls='--', label='avg prediction')
    ax.set_title(f'k = {k}'); ax.legend()
plt.tight_layout(); plt.show()
# k=1: the cloud of fits is wide (variance) but centered on truth (low bias)
# k=25: every fit is nearly identical (low variance) but flattened (bias)

## The U-shaped error curve

In [ ]:
ks = range(1, 31)
bias2, var = [], []
for k in ks:
    preds = np.array([knn_predict(*draw_dataset(), x_grid, k) for _ in range(100)])
    bias2.append(np.mean((preds.mean(0) - true_f(x_grid)) ** 2))
    var.append(preds.var(0).mean())
bias2, var = np.array(bias2), np.array(var)

plt.figure(figsize=(7, 4.5))
plt.plot(ks, bias2, color='#facc15', lw=2, label='bias²')
plt.plot(ks, var, color='#6366f1', lw=2, label='variance')
plt.plot(ks, bias2 + var, color='#f43f5e', lw=2.5, label='bias² + variance')
plt.axvline(ks[np.argmin(bias2 + var)], color='#14b8a6', ls=':', label='best k')
plt.xlabel('k'); plt.ylabel('error'); plt.legend(); plt.show()

**Try it:** increase the noise to 0.6 — the best k moves up (more averaging needed). Shrink the dataset to n=15 — same direction. The optimal complexity depends on data size and noise, never on the model alone.

---
## ✏️ Your turn

Exercise scaffold: estimate bias² and variance **empirically**, exactly as defined — by refitting the same model on many resampled training sets.

### Exercise — Measure bias² and variance of k-NN regression

Draw `n_sets` training sets from `y = sin(2πx) + noise`, fit a k-NN regressor to each, and evaluate all fits on a fixed grid. Then:

- **variance** = average (over grid points) of the variance of predictions across fits
- **bias²** = average squared gap between the *mean* prediction and the true function

You should find: k=1 → variance dominates; large k → bias dominates.

In [ ]:
def knn_regress(x_train, y_train, x_eval, k):
    """1-D k-NN regression: mean y of the k nearest x's, for each x in x_eval."""
    preds = []
    for xe in x_eval:
        nearest = np.argsort(np.abs(x_train - xe))[:k]
        preds.append(y_train[nearest].mean())
    return np.array(preds)


def bias_variance(k, n_sets=30, n_points=30, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    x_eval = np.linspace(0, 1, 50)
    f_true = np.sin(2 * np.pi * x_eval)

    all_preds = []
    for _ in range(n_sets):
        x_tr = rng.random(n_points)
        y_tr = np.sin(2 * np.pi * x_tr) + rng.normal(0, noise, n_points)
        # TODO(you): fit/evaluate this training set on x_eval and collect the result
        ...
    all_preds = np.array(all_preds)          # shape (n_sets, 50)

    # TODO(you): mean prediction at each grid point, shape (50,)
    mean_pred = ...

    # TODO(you): bias² = mean over grid of (mean_pred - f_true)²
    bias2 = ...

    # TODO(you): variance = mean over grid of variance across fits (hint: all_preds.var(axis=0))
    variance = ...

    return bias2, variance

In [ ]:
# Checks — run me
b1, v1 = bias_variance(k=1)
b25, v25 = bias_variance(k=25)

print(f"k=1  : bias² = {b1:.4f}, variance = {v1:.4f}")
print(f"k=25 : bias² = {b25:.4f}, variance = {v25:.4f}")

assert v1 > v25, "small k must have HIGHER variance than large k"
assert b25 > b1, "large k must have HIGHER bias² than small k"
assert v1 > b1, "at k=1, variance should dominate bias²"
assert b25 > v25, "at k=25, bias² should dominate variance"
print("✅ Exercise passed — you just reproduced the tradeoff from the lesson's visualization")

<details>
<summary>💡 Show solution</summary>

```python
def bias_variance(k, n_sets=30, n_points=30, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    x_eval = np.linspace(0, 1, 50)
    f_true = np.sin(2 * np.pi * x_eval)

    all_preds = []
    for _ in range(n_sets):
        x_tr = rng.random(n_points)
        y_tr = np.sin(2 * np.pi * x_tr) + rng.normal(0, noise, n_points)
        all_preds.append(knn_regress(x_tr, y_tr, x_eval, k))
    all_preds = np.array(all_preds)

    mean_pred = all_preds.mean(axis=0)
    bias2 = np.mean((mean_pred - f_true) ** 2)
    variance = np.mean(all_preds.var(axis=0))
    return bias2, variance
```

</details>